# Multiclass Classification

While binary classifiers are used to distinguish between two classes (e.g. detect if a transaction is a fraudulent one, classify an email into either spam or non-spam and etc.), multiclass classifiers distinguish between more than two classes. 

There are various ways that we can use to perform multiclass classification by leveraging any binary classifiers. In this exercise, you will implement two such strategies for multiclass classification: _One-versus-all_ strategy and _One-versus-one_ strategy.

- **One-versus-all (OvA)**: In this strategy, we train a single binary classifier per class, with the samples of that class as positive samples and all other samples as negatives. During inference, we get the prediction from each classifier and select the class with the highest score. This strategy is also called the one-versus-the-rest strtegey. 

- **One-versus-one (OvO)**: In this strategy, we train a binary classifier for every pair of classes. If there are N classes in the problem, you need to train N * (N-1) / 2 classifiers. During inference, we have to run through all N * (N-1) / 2 classifiers and ses which class wins the most votes. The main advantage of OvO strategy is that each binary classifier only needs to be train on the part of the training dataset for the two classes that it needs to separate. 

In [1]:
# import packages
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression

# make this notebook's output stable across runs
np.random.seed(0)

## Avila Dataset

In this lab assignment, we use the [Avila](https://archive.ics.uci.edu/ml/datasets/Avila) data set has been extracted from 800 images of the the "Avila Bible", a giant Latin copy of the whole Bible produced during the XII century between Italy and Spain.  
The palaeographic analysis of the  manuscript has  individuated the presence of 12 copyists. The pages written by each copyist are not equally numerous. 
Each pattern contains 10 features and corresponds to a group of 4 consecutive rows.

The prediction task consists in associating each pattern to one of the 12 copyists (labeled as: A, B, C, D, E, F, G, H, I, W, X, Y).
The data have has been normalized, by using the Z-normalization method, and divided in two data sets: a training set containing 10430 samples, and a test set  containing the 10437 samples.


In [2]:
# Load train and test data from CSV files.
train = pd.read_csv("avila-tr.txt", header=None)
test = pd.read_csv("avila-ts.txt", header=None)

X_train = train.iloc[:,:-1]
y_train = train.iloc[:,-1]

X_test = test.iloc[:,:-1]
y_test = test.iloc[:,-1]

In [3]:
y_train.value_counts()

A    4286
F    1961
E    1095
I     831
X     522
H     519
G     446
D     352
Y     266
C     103
W      44
B       5
Name: 10, dtype: int64

In [4]:
y_test.value_counts()

A    4286
F    1962
E    1095
I     832
X     522
H     520
G     447
D     353
Y     267
C     103
W      45
B       5
Name: 10, dtype: int64

## Question 1.1: Check for missing Data

In [5]:
X_test.isna().sum()

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64

In [6]:
X_train.isna().sum()

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64

## Question 1.2: Apply Z-normalization to data

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train,y_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

## Question 2.1: Write a method to train multiple logistic regression models performing One vs All (OvA) classification. The method allows you to pass in training features, and target. The method returns a list of models and their associated labels. 
### Within the method:
- Determine the list of classes
- Create a place to store all the models
- For each class, train a model with the target variable set to 1 and 0 for all other classes
- Return the list of models trained and associated labels.

In [8]:
def trainOvA(x, y):
    """
    TODO: Train the multiclass classifier using OvA strategy. 
    """
    labels = sorted(y.unique())
    n_labels = len(labels)
    print("Number of classes is {}".format(n_labels))
    models = []
    model_labels = []
    #Create model
    for i in range(n_labels):
        label = labels[i]
        print("Training Logistic Regression model for class {}".format(label))
        model_labels.append(label)
        # update the label according to OvA strategy
        y_OvA = y == label
        y_OvA = y_OvA.replace(to_replace={False,True},value={0,1})
        # Train model
        models.append(LogisticRegression(multi_class='ovr').fit(x,y_OvA))
        
        #models[i] = LogisticRegression(multi_class='ovr').fit(x,y_OvA)
        #model_labels[i] = models[i].feature_names
    return models, model_labels

In [9]:
OvA_models, OvA_labels = trainOvA(X_train,y_train)

Number of classes is 12
Training Logistic Regression model for class A
Training Logistic Regression model for class B
Training Logistic Regression model for class C
Training Logistic Regression model for class D
Training Logistic Regression model for class E
Training Logistic Regression model for class F
Training Logistic Regression model for class G
Training Logistic Regression model for class H
Training Logistic Regression model for class I
Training Logistic Regression model for class W
Training Logistic Regression model for class X
Training Logistic Regression model for class Y


## Question 2.2: Write a method that leverage the multiple models train for OvA, and outputs the majority class.

In [10]:
def predictOvA(models, labels, x):
    """
    TODO: Make predictions on multiclass problems using the OvA strategy. 
    """
    predictions = pd.DataFrame()
    if models == None:
        sys.exit("The model has not been trained yet. Please call train() first. Exiting...")
    
    #Create prediction
    for model in models:
        i = models.index(model)
        predictions[labels[i]]=model.predict(x) #labels used to be OvA_labels
    #predictions = pd.DataFrame(data=predictions)
    return predictions.idxmax(axis=1).value_counts()

In [11]:
predictOvA(OvA_models, OvA_labels, X_test)

A    9307
I     746
X     265
Y      73
E      36
F       5
H       3
B       2
dtype: int64

## Question 2.3: Train OvA model on the Avila dataset

In [12]:
from sklearn.linear_model import LogisticRegression
y_train_OvA = (y_train == "A").replace(to_replace={False,True},value={0,1})
model = LogisticRegression(multi_class='ovr').fit(X_train,y_train_OvA)

## Question 2.4: Predict and evalutate the results of your model

In [13]:
te_z_ova = model.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
y_pred = te_z_ova
y_test_ova = (y_test == "A").replace(to_replace={False,True},value={0,1})
ova_accuracy = accuracy_score(y_test_ova,y_pred) #to do
ova_confuction_matrix = confusion_matrix(y_test_ova,y_pred) #to do
print("Applying the OvA classifier for \"A\" vs Rest")
print("Accuracy of OvA classifier is {}%.".format(round(ova_accuracy*100,2)))
print("Confusion matrix of OvA classifier: \n {}".format(ova_confuction_matrix))

Applying the OvA classifier for "A" vs Rest
Accuracy of OvA classifier is 68.78%.
Confusion matrix of OvA classifier: 
 [[5021 1130]
 [2128 2158]]


---

## Question 3.1: Develop a method that trains a list of models based on the OvO stragety for multiclass classification using logistic regression. 

In [15]:
def trainOvO(x, y):
    """
    TODO: Train the multiclass classifier using OvO strategy. 
    """
    labels = sorted(y.unique())
    n_labels = len(labels)
    n_models = int(len(labels) * (len(labels) - 1) / 2)
    print("Number of classes is {}".format(n_labels))

    models = []#to do
    model_labels = []#to do
    model_idx = 0
    for i in range(n_labels):
        for j in range(i+1, n_labels):
            label_i = labels[i]
            label_j = labels[j]
            print("Training Logistic Regression model to distinguish {} and {}".format(label_i, label_j))
            model_labels.append((label_i,label_j))
            # update the label according to OvA strategy
            train_y = y[(y == label_i)|(y == label_j)]
            train_x = x[train_y.index]

            # construct the logistic regression instance
            lr = LogisticRegression(solver = 'liblinear')
           #don't forget to fit
            models.append(lr.fit(train_x,train_y))
    return models, model_labels

In [16]:
OvO_models, OvO_labels = trainOvO(X_train,y_train)

Number of classes is 12
Training Logistic Regression model to distinguish A and B
Training Logistic Regression model to distinguish A and C
Training Logistic Regression model to distinguish A and D
Training Logistic Regression model to distinguish A and E
Training Logistic Regression model to distinguish A and F
Training Logistic Regression model to distinguish A and G
Training Logistic Regression model to distinguish A and H
Training Logistic Regression model to distinguish A and I
Training Logistic Regression model to distinguish A and W
Training Logistic Regression model to distinguish A and X
Training Logistic Regression model to distinguish A and Y
Training Logistic Regression model to distinguish B and C
Training Logistic Regression model to distinguish B and D
Training Logistic Regression model to distinguish B and E
Training Logistic Regression model to distinguish B and F
Training Logistic Regression model to distinguish B and G
Training Logistic Regression model to distinguis

## Question 3.2: Write a method that leverage the multiple models train for OvO, and outputs the majority class.

In [17]:
def predictOvO(models, labels, x):
    """
    TODO: Make predictions on multiclass problems using the OvO strategy. 
    """
    if models == None:
        sys.exit("The model has not been trained yet. Please call train() first. Exiting...")

    n_models = len(models)
    predictions = pd.DataFrame()
    for model in models:
        i = models.index(model)
        predictions[labels[i]]=model.predict(x)

    return predictions#.mode(axis=1).iloc[:, 0].value_counts()

In [18]:
predictOvO = predictOvO(OvO_models, OvO_labels, X_test)

In [19]:
predictOvO.mode(axis=1).iloc[:, 0].value_counts()

A    7524
I     855
E     568
F     519
X     429
H     252
Y     231
C      27
W      11
G      10
D       6
B       5
Name: 0, dtype: int64

## Question 3.3: Train OvO model on the Avila dataset

In [20]:
models, labels = OvO_models, OvO_labels #already did this in 3.1

## Question 3.4: Predict and evalutate the results of your model

In [21]:
te_z_ovo = predictOvO[("A","I")]

In [22]:

y_pred = te_z_ovo

ovo_confuction_matrix = confusion_matrix(y_test,y_pred)
ovo_accuracy = accuracy_score(y_test,y_pred)

print("Accuracy of OvO classifier is {}%.".format(round(ovo_accuracy*100,2)))
print("Confusion matrix of OvO classifier: \n {}".format(ovo_confuction_matrix))

Accuracy of OvO classifier is 47.98%.
Confusion matrix of OvO classifier: 
 [[4251    0    0    0    0    0    0    0   35    0    0    0]
 [   5    0    0    0    0    0    0    0    0    0    0    0]
 [  99    0    0    0    0    0    0    0    4    0    0    0]
 [ 346    0    0    0    0    0    0    0    7    0    0    0]
 [1083    0    0    0    0    0    0    0   12    0    0    0]
 [1943    0    0    0    0    0    0    0   19    0    0    0]
 [ 447    0    0    0    0    0    0    0    0    0    0    0]
 [ 503    0    0    0    0    0    0    0   17    0    0    0]
 [  75    0    0    0    0    0    0    0  757    0    0    0]
 [  45    0    0    0    0    0    0    0    0    0    0    0]
 [ 390    0    0    0    0    0    0    0  132    0    0    0]
 [  67    0    0    0    0    0    0    0  200    0    0    0]]


## Question 4.1: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) within sklearn supports two approaches for solving multi-class problems: 'ovr', 'multinomial'. Try out both approaches, and evaluate compare the performance agains what you developed in questions 2 and 3.

In [23]:
clf = LogisticRegression(multi_class="ovr").fit(X_train,y_train)
y_ovr = clf.predict(X_test) #predict (to do)

ovr_accuracy = accuracy_score(y_ovr,y_test)
ovr_confuction_matrix = confusion_matrix(y_ovr,y_test)


print("Accuracy of OvO classifier is {}%.".format(round(ovr_accuracy*100,2)))
print("Confusion matrix of OvO classifier: \n {}".format(ovr_confuction_matrix))

Accuracy of OvO classifier is 53.09%.
Confusion matrix of OvO classifier: 
 [[4159    0   80  327  917 1882  413  366   64   34   95   56]
 [   0    5    0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0    0    0]
 [  31    0    9    5   76    4    4   31    2    9   15    1]
 [  62    0   11    6   54   55   23   66    5    0    3    1]
 [   0    0    0    0    0    0    0    0    0    0    0    0]
 [   8    0    0   14   10    4    4   40    3    0    2    0]
 [  18    0    3    0    9   13    0   15  733    0   34   63]
 [   0    0    0    0    0    0    0    0    0    0    0    0]
 [   6    0    0    1   28    0    3    1   15    2  347   20]
 [   2    0    0    0    1    4    0    1   10    0   26  126]]


In [24]:
#class = multinomial
clf = LogisticRegression(multi_class="multinomial", max_iter=300).fit(X_train,y_train)
y_multinomial = clf.predict(X_test)#to do

multinomial_accuracy = accuracy_score(y_multinomial,y_test)#to do
multinomial_confuction_matrix = confusion_matrix(y_multinomial,y_test)#to do


print("Accuracy of OvO classifier is {}%.".format(round(multinomial_accuracy*100,2)))
print("Confusion matrix of OvO classifier: \n {}".format(multinomial_confuction_matrix))

Accuracy of OvO classifier is 56.16%.
Confusion matrix of OvO classifier: 
 [[4041    0   66  288  660 1730  391  305   34   30   49   29]
 [   0    5    0    0    0    0    0    0    0    0    0    0]
 [   1    0    0    0    0    2    0    0    0    0    0    0]
 [   3    0    0    0    0    1    0    0    1    0    0    0]
 [  65    0   13   30  270   15   17   57    4   11   24    3]
 [ 127    0   17   17   60  178   21   47   17    0    3    3]
 [   0    0    0    0    0    0    0    0    0    0    0    0]
 [  18    0    4   14   32   14   14   96    7    0    7    0]
 [  21    0    3    1   11   16    0   13  726    0   23   32]
 [   0    0    0    0    0    0    0    0    0    2    6    0]
 [   7    0    0    3   56    1    4    2   22    2  372   29]
 [   3    0    0    0    6    5    0    0   21    0   38  171]]


## Question 4. Create a new text cell in your Notebook: Complete a 50-100 word summary (or short description of your thinking in applying this week's learning to the solution) of your experience in this assignment. Include:
                                                                      
* What was your incoming experience with this model, if any? 
* What steps you took, what obstacles you encountered.
* How you link this exercise to real-world, machine learning problem-solving. (What steps were missing? What else do you need to learn?) 
> This summary allows your instructor to know how you are doing and allot points for your effort in thinking and planning, and making connections to real-world work.

I've dealt with the sklearn LinearRegression model in the past course but beyond that, I had no prior knowledge of it. At this point I'm somewhat confortable using it and applying it to different scenarios. It's unfortunate I don't have any applied experience with the model but I can only hope to gain it in the future. I did run into some issues when appling OvA, and in creating the methods for both OvA and OvO but in the end they were merely hiccups. The instructions for the lesson were really clear which minimize my confusion throughout the assignment. I could see how this might be applied to scenarios in which one needs to identify the author of something; I imagine this could be useful when trying to track down a specific author maybe in an organization that generates literature, maybe they have a bad habit that needs correcting but due to the nature of the literature it's impossible to immediately find who wrote it. I hope that made sense! I apologize for being unable to give a better real-world example however, this is due to my lack of experience in the field and in applying ML.